# Bias-Variance Tradeoff

Understanding why models fail and how to fix them. Every ML model has errors — bias is systematic errors, variance is sensitivity to training data.

## Total Error = Bias² + Variance + Irreducible Error

- **Bias**: How far predictions are from true values (systematic error)
- **Variance**: How much predictions vary with different training data
- **Irreducible Error**: Noise in data (can't fix)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_squared_error
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

## Underfitting (High Bias)

Model too simple — doesn't capture patterns. Linear model on quadratic data.

High training error, high validation error.

In [ ]:
# Generate quadratic data
X = np.linspace(-3, 3, 50)
y = X**2 + np.random.normal(0, 0.5, len(X))

# Split data
X_train, X_test, y_train, y_test = train_test_split(X.reshape(-1, 1), y, test_size=0.3, random_state=42)

# Underfitting: Linear model on quadratic data
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

# Predictions
y_train_pred_linear = linear_model.predict(X_train)
y_test_pred_linear = linear_model.predict(X_test)

# Calculate errors
train_mse_linear = mean_squared_error(y_train, y_train_pred_linear)
test_mse_linear = mean_squared_error(y_test, y_test_pred_linear)

print(f"Linear Model - Train MSE: {train_mse_linear:.3f}, Test MSE: {test_mse_linear:.3f}")

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_train, y_train, alpha=0.7, label='Training data')
plt.plot(X_train, y_train_pred_linear, color='red', linewidth=2, label='Linear fit')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Underfitting: Linear Model on Quadratic Data')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, alpha=0.7, label='Test data')
plt.plot(X_test, y_test_pred_linear, color='red', linewidth=2, label='Linear predictions')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Underfitting on Test Data')
plt.legend()

plt.tight_layout()
plt.show()

## Overfitting (High Variance)

Model too complex — memorises training data. Perfect training accuracy, poor validation.

Low training error, high validation error.

In [ ]:
# Overfitting: High-degree polynomial
poly_features = PolynomialFeatures(degree=15)
X_train_poly = poly_features.fit_transform(X_train)
X_test_poly = poly_features.transform(X_test)

poly_model = LinearRegression()
poly_model.fit(X_train_poly, y_train)

# Predictions
y_train_pred_poly = poly_model.predict(X_train_poly)
y_test_pred_poly = poly_model.predict(X_test_poly)

# Calculate errors
train_mse_poly = mean_squared_error(y_train, y_train_pred_poly)
test_mse_poly = mean_squared_error(y_test, y_test_pred_poly)

print(f"Polynomial Model - Train MSE: {train_mse_poly:.3f}, Test MSE: {test_mse_poly:.3f}")

# Plot
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.scatter(X_train, y_train, alpha=0.7, label='Training data')
X_plot = np.linspace(-3, 3, 100).reshape(-1, 1)
X_plot_poly = poly_features.transform(X_plot)
y_plot_pred = poly_model.predict(X_plot_poly)
plt.plot(X_plot, y_plot_pred, color='red', linewidth=2, label='Polynomial fit (degree 15)')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Overfitting: Complex Model Memorises Training Data')
plt.legend()

plt.subplot(1, 2, 2)
plt.scatter(X_test, y_test, alpha=0.7, label='Test data')
plt.plot(X_plot, y_plot_pred, color='red', linewidth=2, label='Polynomial predictions')
plt.xlabel('X')
plt.ylabel('y')
plt.title('Overfitting on Test Data - Poor Generalisation')
plt.legend()

plt.tight_layout()
plt.show()

## The Sweet Spot

Balance bias and variance for minimum total error. Use validation curves to find optimal complexity.

In [ ]:
# Validation curve: Model complexity vs error
degrees = range(1, 11)
train_errors = []
val_errors = []

# Use cross-validation approach
from sklearn.model_selection import cross_val_score

for degree in degrees:
    poly_features = PolynomialFeatures(degree=degree)
    X_poly = poly_features.fit_transform(X.reshape(-1, 1))
    
    model = LinearRegression()
    
    # Training error (on full training set)
    model.fit(X_poly, y)
    y_pred_train = model.predict(X_poly)
    train_errors.append(mean_squared_error(y, y_pred_train))
    
    # Validation error (cross-validation)
    val_scores = cross_val_score(model, X_poly, y, cv=5, scoring='neg_mean_squared_error')
    val_errors.append(-val_scores.mean())

# Plot validation curve
plt.figure(figsize=(10, 6))
plt.plot(degrees, train_errors, 'o-', label='Training Error', color='blue')
plt.plot(degrees, val_errors, 'o-', label='Validation Error', color='red')
plt.xlabel('Polynomial Degree (Model Complexity)')
plt.ylabel('Mean Squared Error')
plt.title('Bias-Variance Tradeoff: Validation Curve')
plt.legend()
plt.grid(True, alpha=0.3)

# Find optimal degree
optimal_degree = degrees[np.argmin(val_errors)]
plt.axvline(x=optimal_degree, color='green', linestyle='--', alpha=0.7, 
           label=f'Optimal complexity (degree {optimal_degree})')
plt.legend()

plt.show()

print(f"Optimal polynomial degree: {optimal_degree}")
print(f"Minimum validation error: {min(val_errors):.3f}")

## Cross-Validation

Cross-validation gives reliable estimates of true generalisation performance. Prevents overfitting to a single validation set.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import pandas as pd

# Demonstrate k-fold cross-validation manually
def manual_cross_validation(X, y, model_class, k=5, **model_kwargs):
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    train_scores = []
    val_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        # Split data
        X_train_fold, X_val_fold = X[train_idx], X[val_idx]
        y_train_fold, y_val_fold = y[train_idx], y[val_idx]
        
        # Train model
        model = model_class(**model_kwargs)
        model.fit(X_train_fold, y_train_fold)
        
        # Evaluate
        y_train_pred = model.predict(X_train_fold)
        y_val_pred = model.predict(X_val_fold)
        
        train_scores.append(mean_squared_error(y_train_fold, y_train_pred))
        val_scores.append(mean_squared_error(y_val_fold, y_val_pred))
        
        print(f"Fold {fold+1}: Train MSE = {train_scores[-1]:.3f}, Val MSE = {val_scores[-1]:.3f}")
    
    print(f"\nAverage Train MSE: {np.mean(train_scores):.3f} ± {np.std(train_scores):.3f}")
    print(f"Average Val MSE: {np.mean(val_scores):.3f} ± {np.std(val_scores):.3f}")
    
    return np.mean(val_scores)

# Test different polynomial degrees with cross-validation
degrees = [1, 2, 3, 5, 8, 10]
cv_results = []

for degree in degrees:
    poly_features = PolynomialFeatures(degree=degree)
    X_poly = poly_features.fit_transform(X.reshape(-1, 1))
    
    avg_val_error = manual_cross_validation(X_poly, y, LinearRegression, k=5)
    cv_results.append(avg_val_error)
    print(f"Degree {degree}: CV MSE = {avg_val_error:.3f}\n")

# Plot CV results
plt.figure(figsize=(8, 5))
plt.plot(degrees, cv_results, 'o-', color='purple', linewidth=2)
plt.xlabel('Polynomial Degree')
plt.ylabel('Cross-Validation MSE')
plt.title('Cross-Validation: Finding Optimal Model Complexity')
plt.grid(True, alpha=0.3)
plt.show()

best_degree = degrees[np.argmin(cv_results)]
print(f"Best degree from cross-validation: {best_degree}")

## Key Takeaways

- **High bias** = underfitting = model too simple. Fix: increase complexity.
- **High variance** = overfitting = model too complex. Fix: simplify or regularise.
- Training error decreases as model complexity increases. Validation error forms U-shape.
- More data reduces variance. Better features reduce bias.
- Cross-validation estimates true model performance on unseen data.

## Exercises

1. **Bias vs Variance**: A model has 90% training accuracy but only 70% test accuracy. Is this bias or variance problem? How would you fix it?

2. **Model Complexity**: You're training a decision tree. As you increase max_depth from 1 to 20, what happens to training error? Validation error?

3. **Cross-Validation**: Why is 5-fold CV better than using a single train/validation split?

4. **Real-world**: In fraud detection, would you prefer a model with high precision or high recall? Why?

## Solutions

1. **Bias vs Variance**: This is **variance/overfitting** (high variance). Training error much better than test error. Fix: simplify model, add regularisation, get more data, use dropout.

2. **Model Complexity**: Training error decreases monotonically (gets better). Validation error decreases then increases (U-shape). Optimal depth is where validation error is minimum.

3. **Cross-Validation**: Single split gives noisy estimates (depends on which data goes to validation). 5-fold CV averages over 5 different splits, giving more reliable performance estimate.

4. **Real-world**: High **recall** (catch as many frauds as possible). False negatives (missed fraud) are costly. False positives (investigate legitimate transactions) are less costly.